# CogAttention — Flanker Interference

**Track:** Attention — Selective Attention
**Benchmark:** CogAttention v1.0
**Task:** flanker

---

## Methodology

Tests flanker interference — extracting a target value surrounded by conflicting flanker values. Based on the Eriksen Flanker Task (Eriksen & Eriksen, 1974).

### Cognitive Science Grounding

- **Eriksen Flanker Task** (Eriksen & Eriksen, 1974): a target value is surrounded by conflicting flanker values; the model must extract only the target
- Tests spatial/sequential inhibition of adjacent distractors

### Difficulty Scaling

Easy: 1 flanker, clearly marked target | Medium: 2 flankers | Hard: 3 flankers, subtler marking | Expert: 4 flankers | Frontier: 6+ flankers, minimal target marking

### Scoring

Single assertion — checks whether the model extracted the correct target value while ignoring conflicting flankers.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-FC09BB223676 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Selective Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_flanker(response, gold, kbench):
    gold_val = gold["gold_value"]
    pattern = rf"(?i){_escape_for_regex(gold_val)}"
    kbench.assertions.assert_contains_regex(
        pattern, response,
        expectation=f"Should extract target value '{gold_val}'"
    )


print("CogAttention helpers loaded")
print(f"Task types: ['flanker']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_flanker")
def cogattention_flanker(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention flanker task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_flanker(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "flanker_easy_000",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A preliminary session with Viktor is tentatively set for Friday at 13:00.\n>>> TARGET SENTENCE (2): Olena confirmed the appointment for Friday at 14:30 in room 201.\nSentence 3: A preliminary session with Femi is tentatively set for Thursday at 10:30.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_easy_001",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n>>> TARGET SENTENCE (1): Priya confirmed the appointment for Wednesday at 16:00 in room 305.\nSentence 2: Joaquin suggested meeting on Saturday at 15:00 in the east room instead.\nSentence 3: The briefing with Qadir was moved to Tuesday at 15:00 in the west conference room.\n\nAccording to sentence 1 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_easy_002",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #C-8841 was amended to show 230 units of filters for Kotor.\n>>> TARGET SENTENCE (2): Shipment #D-3506 containing 120 units of filters was dispatched to Zanzibar.\nSentence 3: Manifest #F-9954 was amended to show 350 units of modules for Tallinn.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_easy_003",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n>>> TARGET SENTENCE (1): Manifest #E-7120 lists 120 units of adapters bound for Kumasi.\nSentence 2: The preliminary order #D-3506 allocated 85 units of brackets to Tallinn.\nSentence 3: The preliminary order #F-9954 allocated 420 units of filters to Trieste.\n\nAccording to sentence 1 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_easy_004",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #B-2293 tentatively lists 230 units of filters for Zanzibar.\n>>> TARGET SENTENCE (2): Order #D-3506 for 65 units of modules has been confirmed for delivery to Tallinn.\nSentence 3: A revised order #C-8841 for 85 units of filters is pending approval for Jaipur.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"65\"}"
 },
 {
  "task_id": "flanker_easy_005",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Cancelled order #E-7120 had specified 85 units of modules for Ulaanbaatar.\n>>> TARGET SENTENCE (2): Shipment #B-2293 containing 510 units of panels was dispatched to Oulu.\nSentence 3: A revised order #C-8841 for 230 units of components is pending approval for Kumasi.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_easy_006",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Magnus was moved to Tuesday at 10:30 in the south conference room.\n>>> TARGET SENTENCE (2): The deadline set by Greta falls on Monday, and the review begins at 11:00.\nSentence 3: The briefing with Elio was moved to Monday at 9:00 in the east conference room.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_easy_007",
  "task_type": "flanker",
  "difficulty": "Easy",
  "prompt": "Below are 3 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #F-9954 for 65 units of panels is pending approval for Kumasi.\n>>> TARGET SENTENCE (2): Order #B-2293 for 350 units of modules has been confirmed for delivery to Tbilisi.\nSentence 3: A revised order #F-9954 for 65 units of panels is pending approval for Reykjavik.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_medium_008",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Adaeze requested a reschedule to Thursday at 16:00 in room 201.\nSentence 2: Paloma confirmed the appointment for Friday at 13:00 in room 410.\nSentence 3: Ravi suggested meeting on Wednesday at 15:00 in the south room instead.\nSentence 4: The briefing with Elara was moved to Saturday at 13:00 in the main conference room.\nSentence 5: The deadline proposed by Gael is Monday, with the review starting at 9:00.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_medium_009",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A portable meter at central gave a reading of 67.4 °C.\nSentence 2: An older model gauge at hilltop displayed 45.2 ppm.\nSentence 3: The official reading from riverside showed 67.4 ppm on the certified gauge.\nSentence 4: An uncalibrated device at coastal showed approximately 76.3 mV.\nSentence 5: The secondary sensor near hilltop indicated roughly 23.7 ppm.\n\nAccording to sentence 3 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"67.4 ppm\"}"
 },
 {
  "task_id": "flanker_medium_010",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Freya was moved to Monday at 13:00 in the south conference room.\nSentence 2: Bram confirmed the appointment for Monday at 9:00 in room 201.\nSentence 3: The briefing with Yuki was moved to Monday at 16:00 in the west conference room.\nSentence 4: Qadir requested a reschedule to Saturday at 14:30 in room 305.\nSentence 5: A preliminary session with Xander is tentatively set for Wednesday at 16:00.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_medium_011",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Joaquin suggested meeting on Friday at 11:00 in the west room instead.\nSentence 2: Wren confirmed the appointment for Friday at 15:00 in room 305.\nSentence 3: The deadline proposed by Ravi is Wednesday, with the review starting at 10:30.\nSentence 4: The follow-up with Qadir was postponed to Wednesday at 9:00 in room 603.\nSentence 5: The follow-up with Priya was postponed to Tuesday at 15:00 in room 507.\n\nAccording to sentence 2 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_medium_012",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Shipment #E-7120 with 420 units of panels was redirected to Recife.\nSentence 2: Shipment #D-3506 containing 120 units of panels was dispatched to Recife.\nSentence 3: Cancelled order #F-9954 had specified 85 units of panels for Cartagena.\nSentence 4: Manifest #C-8841 was amended to show 65 units of adapters for Bruges.\nSentence 5: A revised order #B-2293 for 230 units of filters is pending approval for Cartagena.\n\nAccording to sentence 2 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_medium_013",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Kaia was moved to Saturday at 10:30 in the north conference room.\nSentence 2: The briefing with Joelle was moved to Friday at 11:00 in the north conference room.\nSentence 3: Soren confirmed the appointment for Wednesday at 14:30 in room 603.\nSentence 4: Zora requested a reschedule to Wednesday at 14:30 in room 603.\nSentence 5: A preliminary session with Amara is tentatively set for Friday at 15:00.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_medium_014",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #E-7120 for 175 units of modules is pending approval for Ulaanbaatar.\nSentence 2: The preliminary order #E-7120 allocated 85 units of filters to Trieste.\nSentence 3: Shipment #F-9954 containing 175 units of adapters was dispatched to Cusco.\nSentence 4: Manifest #F-9954 was amended to show 420 units of adapters for Kumasi.\nSentence 5: Draft manifest #A-4017 tentatively lists 350 units of filters for Jaipur.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"175\"}"
 },
 {
  "task_id": "flanker_medium_015",
  "task_type": "flanker",
  "difficulty": "Medium",
  "prompt": "Below are 5 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A portable meter at coastal gave a reading of 76.3 mg/L.\nSentence 2: The official reading from downtown showed 14.6 mV on the certified gauge.\nSentence 3: The secondary sensor near riverside indicated roughly 23.7 mV.\nSentence 4: An older model gauge at coastal displayed 67.4 kPa.\nSentence 5: A portable meter at northern gave a reading of 67.4 mg/L.\n\nAccording to sentence 2 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"14.6 mV\"}"
 },
 {
  "task_id": "flanker_hard_016",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Vesna was postponed to Tuesday at 10:30 in room 410.\nSentence 2: A preliminary session with Orla is tentatively set for Monday at 10:30.\nSentence 3: Olena confirmed the appointment for Wednesday at 11:00 in room 112.\nSentence 4: Zora requested a reschedule to Friday at 10:30 in room 201.\nSentence 5: Zain requested a reschedule to Saturday at 16:00 in room 410.\nSentence 6: A preliminary session with Viktor is tentatively set for Wednesday at 10:30.\nSentence 7: The follow-up with Ines was postponed to Saturday at 9:00 in room 507.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Wednesday\"}"
 },
 {
  "task_id": "flanker_hard_017",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A preliminary session with Willa is tentatively set for Wednesday at 11:00.\nSentence 2: The follow-up with Joaquin was postponed to Monday at 11:00 in room 201.\nSentence 3: A preliminary session with Kenji is tentatively set for Thursday at 16:00.\nSentence 4: Xander requested a reschedule to Saturday at 11:00 in room 201.\nSentence 5: Amara confirmed the appointment for Saturday at 14:30 in room 410.\nSentence 6: The follow-up with Vesna was postponed to Thursday at 11:00 in room 603.\nSentence 7: Bashir suggested meeting on Wednesday at 14:30 in the south room instead.\n\nAccording to sentence 5 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_hard_018",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Runa was moved to Wednesday at 10:30 in the north conference room.\nSentence 2: The deadline proposed by Ines is Thursday, with the review starting at 15:00.\nSentence 3: The deadline set by Tariq falls on Monday, and the review begins at 16:00.\nSentence 4: The briefing with Bashir was moved to Thursday at 13:00 in the west conference room.\nSentence 5: The follow-up with Joelle was postponed to Thursday at 10:30 in room 305.\nSentence 6: Ravi requested a reschedule to Wednesday at 9:00 in room 305.\nSentence 7: Zain requested a reschedule to Saturday at 9:00 in room 201.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Monday\"}"
 },
 {
  "task_id": "flanker_hard_019",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: An older model gauge at northern displayed 67.4 lux.\nSentence 2: An older model gauge at downtown displayed 14.6 ppm.\nSentence 3: An older model gauge at downtown displayed 31.8 mV.\nSentence 4: An uncalibrated device at coastal showed approximately 18.9 mV.\nSentence 5: According to the verified sensor at northern, the measurement was 23.7 °C.\nSentence 6: The temporary sensor installed at northern read 23.7 °C.\nSentence 7: A portable meter at northern gave a reading of 52.1 mg/L.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"23.7 °C\"}"
 },
 {
  "task_id": "flanker_hard_020",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #A-4017 was amended to show 230 units of filters for Cartagena.\nSentence 2: The preliminary order #E-7120 allocated 420 units of components to Ulaanbaatar.\nSentence 3: Shipment #B-2293 containing 350 units of filters was dispatched to Kotor.\nSentence 4: The preliminary order #A-4017 allocated 65 units of brackets to Mandalay.\nSentence 5: Cancelled order #C-8841 had specified 65 units of adapters for Valetta.\nSentence 6: Manifest #E-7120 was amended to show 510 units of adapters for Zanzibar.\nSentence 7: Cancelled order #D-3506 had specified 230 units of panels for Recife.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_hard_021",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Greta was postponed to Thursday at 10:30 in room 201.\nSentence 2: The briefing with Soren was moved to Thursday at 10:30 in the east conference room.\nSentence 3: The meeting with Gael is scheduled for Saturday at 9:00 in the north conference room.\nSentence 4: The deadline proposed by Wren is Thursday, with the review starting at 13:00.\nSentence 5: The follow-up with Idris was postponed to Monday at 10:30 in room 410.\nSentence 6: The follow-up with Ravi was postponed to Wednesday at 11:00 in room 603.\nSentence 7: The deadline proposed by Vesna is Wednesday, with the review starting at 9:00.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_hard_022",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: A revised order #F-9954 for 175 units of filters is pending approval for Cusco.\nSentence 2: Manifest #A-4017 was amended to show 230 units of brackets for Ulaanbaatar.\nSentence 3: The preliminary order #A-4017 allocated 65 units of components to Kumasi.\nSentence 4: Shipment #C-8841 containing 510 units of modules was dispatched to Bruges.\nSentence 5: A revised order #D-3506 for 65 units of components is pending approval for Fez.\nSentence 6: Shipment #C-8841 with 510 units of modules was redirected to Tallinn.\nSentence 7: Manifest #C-8841 was amended to show 85 units of panels for Fez.\n\nAccording to sentence 4 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_hard_023",
  "task_type": "flanker",
  "difficulty": "Hard",
  "prompt": "Below are 7 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The follow-up with Kaia was postponed to Friday at 13:00 in room 112.\nSentence 2: The briefing with Freya was moved to Saturday at 10:30 in the east conference room.\nSentence 3: Ravi confirmed the appointment for Friday at 15:00 in room 603.\nSentence 4: The briefing with Amara was moved to Saturday at 15:00 in the main conference room.\nSentence 5: Soren suggested meeting on Wednesday at 9:00 in the north room instead.\nSentence 6: The briefing with Ines was moved to Monday at 9:00 in the south conference room.\nSentence 7: Tariq suggested meeting on Friday at 11:00 in the south room instead.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Friday\"}"
 },
 {
  "task_id": "flanker_expert_024",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The secondary sensor near central indicated roughly 45.2 mg/L.\nSentence 2: An uncalibrated device at riverside showed approximately 14.6 kPa.\nSentence 3: The backup instrument at riverside registered 18.9 lux before recalibration.\nSentence 4: According to the verified sensor at riverside, the measurement was 67.4 kPa.\nSentence 5: An uncalibrated device at downtown showed approximately 31.8 lux.\nSentence 6: A portable meter at riverside gave a reading of 31.8 kPa.\nSentence 7: A portable meter at downtown gave a reading of 31.8 kPa.\nSentence 8: The temporary sensor installed at downtown read 67.4 mg/L.\nSentence 9: The secondary sensor near coastal indicated roughly 45.2 kPa.\n\nAccording to sentence 4 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"67.4 kPa\"}"
 },
 {
  "task_id": "flanker_expert_025",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Shipment #F-9954 with 420 units of panels was redirected to Gdansk.\nSentence 2: The preliminary order #F-9954 allocated 350 units of panels to Valetta.\nSentence 3: Order #B-2293 for 120 units of components has been confirmed for delivery to Oulu.\nSentence 4: Draft manifest #A-4017 tentatively lists 85 units of filters for Luang Prabang.\nSentence 5: The preliminary order #F-9954 allocated 350 units of panels to Mandalay.\nSentence 6: Shipment #F-9954 with 350 units of brackets was redirected to Mandalay.\nSentence 7: Cancelled order #C-8841 had specified 420 units of modules for Tbilisi.\nSentence 8: The preliminary order #A-4017 allocated 230 units of adapters to Cartagena.\nSentence 9: Cancelled order #E-7120 had specified 230 units of adapters for Ulaanbaatar.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"120\"}"
 },
 {
  "task_id": "flanker_expert_026",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Celine suggested meeting on Thursday at 14:30 in the north room instead.\nSentence 2: The deadline proposed by Celine is Friday, with the review starting at 13:00.\nSentence 3: The deadline set by Joaquin falls on Saturday, and the review begins at 10:30.\nSentence 4: Lumi requested a reschedule to Saturday at 13:00 in room 305.\nSentence 5: The briefing with Zora was moved to Friday at 14:30 in the main conference room.\nSentence 6: Tala requested a reschedule to Saturday at 16:00 in room 410.\nSentence 7: The follow-up with Hana was postponed to Friday at 9:00 in room 603.\nSentence 8: The deadline proposed by Ravi is Monday, with the review starting at 14:30.\nSentence 9: Lumi requested a reschedule to Friday at 13:00 in room 507.\n\nAccording to sentence 3 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_expert_027",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #A-4017 tentatively lists 120 units of filters for Tbilisi.\nSentence 2: Cancelled order #D-3506 had specified 120 units of components for Ulaanbaatar.\nSentence 3: Shipment #D-3506 with 510 units of adapters was redirected to Luang Prabang.\nSentence 4: Draft manifest #E-7120 tentatively lists 420 units of modules for Plovdiv.\nSentence 5: A revised order #C-8841 for 65 units of components is pending approval for Gdansk.\nSentence 6: Shipment #D-3506 containing 350 units of panels was dispatched to Tbilisi.\nSentence 7: Cancelled order #E-7120 had specified 120 units of brackets for Luang Prabang.\nSentence 8: Manifest #B-2293 was amended to show 350 units of panels for Recife.\nSentence 9: Shipment #B-2293 with 120 units of brackets was redirected to Cusco.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_expert_028",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Manifest #F-9954 was amended to show 175 units of filters for Reykjavik.\nSentence 2: The preliminary order #E-7120 allocated 120 units of modules to Ulaanbaatar.\nSentence 3: Draft manifest #D-3506 tentatively lists 85 units of components for Jaipur.\nSentence 4: Manifest #D-3506 was amended to show 175 units of panels for Valetta.\nSentence 5: Draft manifest #A-4017 tentatively lists 175 units of components for Gdansk.\nSentence 6: Shipment #A-4017 containing 510 units of brackets was dispatched to Trieste.\nSentence 7: The preliminary order #E-7120 allocated 85 units of components to Luang Prabang.\nSentence 8: The preliminary order #E-7120 allocated 65 units of components to Cartagena.\nSentence 9: Cancelled order #E-7120 had specified 350 units of adapters for Tallinn.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_expert_029",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #B-2293 tentatively lists 510 units of filters for Cusco.\nSentence 2: Manifest #B-2293 was amended to show 85 units of modules for Jaipur.\nSentence 3: Draft manifest #A-4017 tentatively lists 230 units of adapters for Mandalay.\nSentence 4: A revised order #D-3506 for 65 units of components is pending approval for Reykjavik.\nSentence 5: Shipment #B-2293 containing 420 units of filters was dispatched to Mandalay.\nSentence 6: A revised order #A-4017 for 85 units of brackets is pending approval for Ulaanbaatar.\nSentence 7: The preliminary order #B-2293 allocated 120 units of adapters to Plovdiv.\nSentence 8: Draft manifest #C-8841 tentatively lists 420 units of adapters for Luang Prabang.\nSentence 9: A revised order #C-8841 for 350 units of adapters is pending approval for Mandalay.\n\nAccording to sentence 5 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"420\"}"
 },
 {
  "task_id": "flanker_expert_030",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about schedule. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: The briefing with Haruto was moved to Monday at 9:00 in the north conference room.\nSentence 2: The briefing with Nico was moved to Wednesday at 11:00 in the main conference room.\nSentence 3: The briefing with Nalini was moved to Monday at 15:00 in the south conference room.\nSentence 4: The deadline set by Dmitri falls on Saturday, and the review begins at 16:00.\nSentence 5: Greta suggested meeting on Friday at 11:00 in the north room instead.\nSentence 6: A preliminary session with Dariush is tentatively set for Saturday at 10:30.\nSentence 7: A preliminary session with Nalini is tentatively set for Monday at 16:00.\nSentence 8: The deadline proposed by Zora is Thursday, with the review starting at 10:30.\nSentence 9: The follow-up with Qadir was postponed to Friday at 11:00 in room 603.\n\nAccording to sentence 4 ONLY, what day is the event scheduled for?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"Saturday\"}"
 },
 {
  "task_id": "flanker_expert_031",
  "task_type": "flanker",
  "difficulty": "Expert",
  "prompt": "Below are 9 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\nSentence 1: Draft manifest #A-4017 tentatively lists 65 units of adapters for Oulu.\nSentence 2: Manifest #B-2293 was amended to show 350 units of components for Zanzibar.\nSentence 3: Order #A-4017 for 350 units of modules has been confirmed for delivery to Reykjavik.\nSentence 4: The preliminary order #A-4017 allocated 65 units of adapters to Luang Prabang.\nSentence 5: A revised order #B-2293 for 65 units of components is pending approval for Kumasi.\nSentence 6: Manifest #F-9954 was amended to show 65 units of components for Zanzibar.\nSentence 7: The preliminary order #A-4017 allocated 65 units of panels to Tbilisi.\nSentence 8: The preliminary order #E-7120 allocated 350 units of components to Tallinn.\nSentence 9: Shipment #D-3506 with 85 units of adapters was redirected to Tallinn.\n\nAccording to sentence 3 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_032",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Cancelled order #E-7120 had specified 420 units of modules for Luang Prabang.\n2. Shipment #F-9954 with 350 units of panels was redirected to Gdansk.\n3. Draft manifest #A-4017 tentatively lists 120 units of panels for Bruges.\n4. Shipment #D-3506 with 120 units of adapters was redirected to Recife.\n5. Shipment #B-2293 with 510 units of modules was redirected to Ulaanbaatar.\n6. Shipment #B-2293 with 350 units of adapters was redirected to Reykjavik.\n7. Draft manifest #D-3506 tentatively lists 120 units of components for Gdansk.\n8. Manifest #A-4017 lists 510 units of modules bound for Zanzibar.\n9. The preliminary order #F-9954 allocated 65 units of filters to Bruges.\n10. The preliminary order #E-7120 allocated 420 units of filters to Bruges.\n11. Manifest #F-9954 was amended to show 420 units of filters for Gdansk.\n12. Draft manifest #F-9954 tentatively lists 230 units of adapters for Tallinn.\n13. A revised order #B-2293 for 175 units of panels is pending approval for Ulaanbaatar.\n\nAccording to sentence 8 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"510\"}"
 },
 {
  "task_id": "flanker_frontier_033",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. An older model gauge at central displayed 14.6 °C.\n2. An older model gauge at downtown displayed 45.2 kPa.\n3. The backup instrument at downtown registered 45.2 kPa before recalibration.\n4. The temporary sensor installed at coastal read 23.7 kPa.\n5. The calibrated instrument recorded a reading of 14.6 mV at the coastal station.\n6. A portable meter at northern gave a reading of 18.9 mV.\n7. An older model gauge at northern displayed 67.4 kPa.\n8. An older model gauge at central displayed 31.8 mg/L.\n9. The secondary sensor near northern indicated roughly 52.1 mV.\n10. The backup instrument at coastal registered 23.7 lux before recalibration.\n11. An older model gauge at downtown displayed 14.6 ppm.\n12. An older model gauge at central displayed 52.1 kPa.\n13. The temporary sensor installed at coastal read 52.1 mV.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"14.6 mV\"}"
 },
 {
  "task_id": "flanker_frontier_034",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Draft manifest #E-7120 tentatively lists 420 units of adapters for Cartagena.\n2. Manifest #B-2293 was amended to show 120 units of modules for Trieste.\n3. The preliminary order #E-7120 allocated 350 units of brackets to Gdansk.\n4. The preliminary order #B-2293 allocated 65 units of brackets to Tallinn.\n5. Manifest #B-2293 was amended to show 85 units of modules for Luang Prabang.\n6. Shipment #E-7120 containing 350 units of filters was dispatched to Luang Prabang.\n7. A revised order #F-9954 for 420 units of adapters is pending approval for Bruges.\n8. Manifest #F-9954 was amended to show 350 units of components for Luang Prabang.\n9. A revised order #B-2293 for 85 units of components is pending approval for Recife.\n10. Cancelled order #A-4017 had specified 510 units of brackets for Cartagena.\n11. Manifest #F-9954 was amended to show 230 units of filters for Kumasi.\n12. A revised order #C-8841 for 120 units of adapters is pending approval for Kotor.\n13. Cancelled order #E-7120 had specified 420 units of adapters for Trieste.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_035",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Cancelled order #C-8841 had specified 65 units of modules for Tbilisi.\n2. Cancelled order #D-3506 had specified 175 units of filters for Bruges.\n3. A revised order #F-9954 for 120 units of filters is pending approval for Mandalay.\n4. Cancelled order #D-3506 had specified 510 units of components for Jaipur.\n5. Shipment #B-2293 with 85 units of modules was redirected to Oulu.\n6. Manifest #B-2293 lists 350 units of modules bound for Reykjavik.\n7. Manifest #B-2293 was amended to show 230 units of filters for Trieste.\n8. Shipment #E-7120 with 85 units of panels was redirected to Plovdiv.\n9. A revised order #C-8841 for 120 units of panels is pending approval for Plovdiv.\n10. The preliminary order #A-4017 allocated 85 units of adapters to Kotor.\n11. Shipment #E-7120 with 510 units of brackets was redirected to Fez.\n12. Shipment #E-7120 with 420 units of filters was redirected to Fez.\n13. Cancelled order #E-7120 had specified 120 units of panels for Jaipur.\n\nAccording to sentence 6 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_036",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Manifest #E-7120 was amended to show 230 units of adapters for Trieste.\n2. Cancelled order #B-2293 had specified 420 units of components for Gdansk.\n3. Manifest #F-9954 was amended to show 65 units of components for Trieste.\n4. Manifest #F-9954 was amended to show 420 units of brackets for Zanzibar.\n5. Manifest #F-9954 was amended to show 65 units of brackets for Ulaanbaatar.\n6. Shipment #A-4017 with 350 units of components was redirected to Valetta.\n7. Shipment #B-2293 containing 85 units of panels was dispatched to Tallinn.\n8. A revised order #B-2293 for 175 units of panels is pending approval for Ulaanbaatar.\n9. Shipment #E-7120 with 230 units of components was redirected to Zanzibar.\n10. A revised order #E-7120 for 65 units of filters is pending approval for Ulaanbaatar.\n11. Draft manifest #B-2293 tentatively lists 350 units of components for Recife.\n12. Draft manifest #D-3506 tentatively lists 85 units of modules for Trieste.\n13. Cancelled order #F-9954 had specified 350 units of filters for Luang Prabang.\n\nAccording to sentence 7 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"85\"}"
 },
 {
  "task_id": "flanker_frontier_037",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. A portable meter at downtown gave a reading of 76.3 mV.\n2. An older model gauge at downtown displayed 31.8 kPa.\n3. An older model gauge at riverside displayed 23.7 mV.\n4. An older model gauge at northern displayed 67.4 °C.\n5. The temporary sensor installed at central read 76.3 ppm.\n6. The temporary sensor installed at northern read 23.7 kPa.\n7. The official reading from downtown showed 52.1 ppm on the certified gauge.\n8. The secondary sensor near riverside indicated roughly 18.9 °C.\n9. The secondary sensor near hilltop indicated roughly 31.8 kPa.\n10. An older model gauge at central displayed 18.9 kPa.\n11. The temporary sensor installed at central read 23.7 mg/L.\n12. The backup instrument at downtown registered 18.9 lux before recalibration.\n13. An older model gauge at central displayed 18.9 mg/L.\n\nAccording to sentence 7 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"52.1 ppm\"}"
 },
 {
  "task_id": "flanker_frontier_038",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about shipment. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. Draft manifest #A-4017 tentatively lists 175 units of adapters for Trieste.\n2. Cancelled order #C-8841 had specified 120 units of filters for Cusco.\n3. Cancelled order #B-2293 had specified 350 units of panels for Tallinn.\n4. The preliminary order #C-8841 allocated 350 units of filters to Gdansk.\n5. The preliminary order #C-8841 allocated 230 units of filters to Recife.\n6. A revised order #F-9954 for 420 units of adapters is pending approval for Mandalay.\n7. The preliminary order #C-8841 allocated 120 units of filters to Trieste.\n8. Manifest #B-2293 lists 350 units of modules bound for Fez.\n9. Manifest #B-2293 was amended to show 65 units of modules for Bruges.\n10. Shipment #D-3506 with 350 units of adapters was redirected to Zanzibar.\n11. Cancelled order #E-7120 had specified 175 units of brackets for Kotor.\n12. The preliminary order #A-4017 allocated 120 units of adapters to Recife.\n13. Manifest #D-3506 was amended to show 65 units of components for Bruges.\n\nAccording to sentence 8 ONLY, how many units were included?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"350\"}"
 },
 {
  "task_id": "flanker_frontier_039",
  "task_type": "flanker",
  "difficulty": "Frontier",
  "prompt": "Below are 13 similar sentences about measurement. Most contain different values. Answer the question based on ONLY the specified sentence.\n\n1. The backup instrument at downtown registered 52.1 lux before recalibration.\n2. The secondary sensor near riverside indicated roughly 31.8 °C.\n3. The secondary sensor near central indicated roughly 23.7 kPa.\n4. An uncalibrated device at hilltop showed approximately 14.6 ppm.\n5. According to the verified sensor at central, the measurement was 31.8 kPa.\n6. The backup instrument at coastal registered 76.3 mV before recalibration.\n7. An older model gauge at riverside displayed 23.7 mg/L.\n8. A portable meter at riverside gave a reading of 18.9 mg/L.\n9. An uncalibrated device at downtown showed approximately 67.4 lux.\n10. The backup instrument at downtown registered 45.2 mg/L before recalibration.\n11. The temporary sensor installed at central read 31.8 mV.\n12. An older model gauge at coastal displayed 52.1 mV.\n13. The backup instrument at riverside registered 52.1 kPa before recalibration.\n\nAccording to sentence 5 ONLY, what was the exact measurement reading?\n\nANSWER: [your answer]",
  "gold_json": "{\"gold_value\": \"31.8 kPa\"}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['flanker']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "flanker": cogattention_flanker,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Selective Attention")
